# The Grid — Ollama Backend (Colab + Google Drive)

**Runtime → Change runtime type → T4 GPU** before running.

Models are stored on your Google Drive so they only download once (~8GB total).

At the end you get a public URL — paste it into your `.env` as `OLLAMA_BASE_URL`.

In [ ]:
# Cell 1 — Mount Google Drive
from google.colab import drive
drive.mount('/gdrive')

import os
MODEL_DIR = '/gdrive/MyDrive/ollama_models'
os.makedirs(MODEL_DIR, exist_ok=True)
print(f'Model cache: {MODEL_DIR}')

In [ ]:
# Cell 2 — Install Ollama + symlink models to Google Drive
!curl -fsSL https://ollama.com/install.sh | sh

# Symlink ~/.ollama/models → Google Drive so models persist
!mkdir -p /root/.ollama
!rm -rf /root/.ollama/models
!ln -s /gdrive/MyDrive/ollama_models /root/.ollama/models
print('Symlink: ~/.ollama/models → Google Drive')

In [ ]:
# Cell 3 — Start Ollama server
import subprocess, time

proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(4)

import urllib.request
try:
    urllib.request.urlopen('http://localhost:11434')
    print('Ollama running on :11434')
except:
    print('Ollama not responding — re-run this cell')

In [ ]:
# Cell 4 — Pull models (skips if already on Drive)
import subprocess, os

MODELS = [
    'dolphin-mistral:latest',   # uncensored general — 4.1GB
    'deepseek-coder:6.7b',      # uncensored coder   — 3.8GB
]

for model in MODELS:
    slug = model.replace(':', '-').replace('/', '-')
    marker = f'/gdrive/MyDrive/ollama_models/.pulled_{slug}'
    if os.path.exists(marker):
        print(f'[skip] {model} already on Drive')
        continue
    print(f'[pull] {model} ...')
    result = subprocess.run(['ollama', 'pull', model], capture_output=True, text=True)
    if result.returncode == 0:
        open(marker, 'w').close()
        print(f'[done] {model}')
    else:
        print(f'[fail] {model}\n{result.stderr}')

In [ ]:
# Cell 5 — Expose via ngrok (survives signal drops, auto-reconnects)
!pip install -q pyngrok
from pyngrok import ngrok, conf
import getpass

# Free token at https://ngrok.com (Dashboard → Your Authtoken)
token = getpass.getpass('Paste your ngrok authtoken: ')
conf.get_default().auth_token = token

# Kill any existing tunnels then open a fresh one
ngrok.kill()
public_url = ngrok.connect(11434, bind_tls=True)

print('\n' + '='*60)
print(f'  OLLAMA_BASE_URL={public_url}')
print(f'  OLLAMA_MODEL=dolphin-mistral:latest')
print('  Paste both lines into your .env then restart the server')
print('='*60)

In [ ]:
# Cell 6 — Keep alive (run last, leave running)
# Prevents Colab from killing the idle session
import time
print('Keeping session alive. Stop this cell to shut down.')
while True:
    time.sleep(60)
    print('.', end='', flush=True)

## Switching models mid-session

Change `OLLAMA_MODEL` in `.env` to `deepseek-coder:6.7b` and restart the Node server.

## Models stored on your Drive after first run

| Model | Size | Use for |
|-------|------|---------|
| `dolphin-mistral:latest` | 4.1GB | Characters (Nina, Iris, Vale, Hazel) |
| `deepseek-coder:6.7b` | 3.8GB | Code tasks |

## Cost
- Compute: **$0** (Colab free T4)
- Storage: **~8GB** of your 8TB Google Drive
- ngrok free tier: **1 tunnel, stable URL per session**
- Session limit: ~12h, then re-run Cell 5 and paste the new URL into `.env`